In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
###########################################################
# helper function
import torch
import numpy as np
import random
import torchaudio
import os
import glob
from pathlib import Path

# --- SET YOUR KAGGLE PATHS ---
INPUT_BASE = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')


def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)







def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
    genres = ["blues", "classical", "country", "disco", "hiphop",
"jazz", "metal", "pop", "reggae", "rock"
]
    target_length = target_sr * duration
    
    # Get noise files from read-only input
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        # Create output directories in the writable /kaggle/working/ directory
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders: 
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Basic Resampling check (if needed)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                    
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Save to /kaggle/working/
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)

# Run the generation
generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)




import os
import glob
import torch
import torchaudio
from pathlib import Path

def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    # Find all .wav files in the input directory
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    
    if not wav_files:
        print(f"Warning: No .wav files found in {input_dir}")
        return

    for wav_path in wav_files:
        # Replicate directory structure
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        
        # Ensure the target directory exists in /kaggle/working/
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Process and save
        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        
        torch.save(mel_spec_db, out_path)
    
    print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")


INPUT_DIR = '/kaggle/working/synthetic_mashups/train'
OUTPUT_DIR = '/kaggle/working/features/train'

extract_and_save_features(INPUT_DIR, OUTPUT_DIR)

Successfully saved 500 feature files to /kaggle/working/features/train


In [3]:
#########################################################################
waveform, sr = torchaudio.load("/kaggle/working/synthetic_mashups/train/blues/mashup_000.wav")
print(waveform.shape)

torch.Size([2, 661500])


In [4]:
#############################################################################
f = torch.load('/kaggle/working/features/train/blues/mashup_000.pt')
print(f.shape)

torch.Size([2, 128, 1292])


In [5]:
##########################################################
#question 4 
import torch
import torch.nn as nn

class CRNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(

            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

    def forward(self, x):

        x = self.cnn(x)

        # shape right after second MaxPool2d
        print("Shape after CNN:", x.shape)

        return x


# create model
model = CRNN()

# simulate batch input
x = torch.randn(32, 1, 128, 1292)

# run forward pass
output = model(x)

Shape after CNN: torch.Size([32, 64, 32, 323])


In [6]:
####################################################################################
import torch
import torch.nn as nn
import glob
import os
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

# ==============================
# Dataset
# ==============================

class PrecomputedFeatureDataset(Dataset):
    def __init__(self, features_dir):
        self.files = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)

        self.genres = sorted([
            'blues','classical','country','disco','hiphop',
            'jazz','metal','pop','reggae','rock'
        ])

        self.genre_to_idx = {g:i for i,g in enumerate(self.genres)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        file_path = self.files[idx]

        genre = Path(file_path).parent.name
        label = self.genre_to_idx[genre]

        feature = torch.load(file_path)

        # convert stereo → mono
        if feature.shape[0] == 2:
            feature = feature.mean(dim=0, keepdim=True)

        # ensure channel dimension exists
        if feature.dim() == 2:
            feature = feature.unsqueeze(0)

        return feature, label


# ==============================
# CRNN Model
# ==============================

class CRNN(nn.Module):

    def __init__(self, num_classes=10):
        super().__init__()

        # CNN Backbone
        self.cnn = nn.Sequential(

            # Block 1
            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # LSTM
        # After CNN:
        # channels = 64
        # mels = 128 / 4 = 32
        # features = 64 * 32 = 2048

        self.lstm = nn.LSTM(
            input_size=2048,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Final classifier
        self.fc = nn.Linear(
            in_features=128,   # 64 * 2 because bidirectional
            out_features=num_classes
        )

    def forward(self, x):

        # x shape
        # (Batch, 1, 128, Time)

        x = self.cnn(x)

        # shape
        # (Batch, 64, 32, Time)

        b, c, f, t = x.shape

        # move time dimension forward
        x = x.permute(0, 3, 1, 2)

        # (Batch, Time, Channels, Mels)

        x = x.reshape(b, t, c * f)

        # (Batch, Time, 2048)

        x, _ = self.lstm(x)

        # (Batch, Time, 128)

        x, _ = torch.max(x, dim=1)

        # (Batch, 128)

        logits = self.fc(x)

        return logits


# ==============================
# Create Dataset and DataLoader
# ==============================

FEATURE_DIR = "/kaggle/working/features/train"

dataset = PrecomputedFeatureDataset(FEATURE_DIR)

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2
)


# ==============================
# Initialize Model
# ==============================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CRNN(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)


# ==============================
# Test Forward Pass
# ==============================

sample_batch, labels = next(iter(loader))

sample_batch = sample_batch.to(device)

output = model(sample_batch)

print("Input shape :", sample_batch.shape)
print("Output shape:", output.shape)

Input shape : torch.Size([16, 1, 128, 1292])
Output shape: torch.Size([16, 10])


In [7]:
######################################################################
#question-5
import torch
import torch.nn as nn

lstm = nn.LSTM(
    input_size=2048,
    hidden_size=64,
    num_layers=1,
    batch_first=True,
    bidirectional=True
)

num_params = sum(p.numel() for p in lstm.parameters() if p.requires_grad)

print(num_params)

1082368


In [8]:
#####################################################
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
num_epochs = 10
for epoch in range(num_epochs):

    # -------------------------
    # TRAINING
    # -------------------------
    model.train()

    train_loss = 0
    correct = 0
    total = 0

    for features, labels in train_loader:

        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(features)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        _, preds = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

    train_acc = 100 * correct / total


    # -------------------------
    # VALIDATION
    # -------------------------
    model.eval()

    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for features, labels in val_loader:

            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    val_acc = 100 * correct / total


    # -------------------------
    # PRINT RESULTS
    # -------------------------
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.2f}%")
    print()

Epoch [1/10]
Train Loss: 2.1866 | Train Acc: 24.00%
Val Loss: 2.1520 | Val Acc: 28.00%

Epoch [2/10]
Train Loss: 1.9157 | Train Acc: 43.25%
Val Loss: 2.0241 | Val Acc: 36.00%

Epoch [3/10]
Train Loss: 1.7790 | Train Acc: 45.25%
Val Loss: 1.9408 | Val Acc: 40.00%

Epoch [4/10]
Train Loss: 1.6408 | Train Acc: 53.50%
Val Loss: 1.8090 | Val Acc: 40.00%

Epoch [5/10]
Train Loss: 1.5416 | Train Acc: 54.75%
Val Loss: 1.7347 | Val Acc: 50.00%

Epoch [6/10]
Train Loss: 1.4528 | Train Acc: 60.75%
Val Loss: 1.7060 | Val Acc: 47.00%

Epoch [7/10]
Train Loss: 1.3555 | Train Acc: 66.50%
Val Loss: 1.6300 | Val Acc: 52.00%

Epoch [8/10]
Train Loss: 1.2415 | Train Acc: 70.50%
Val Loss: 1.5436 | Val Acc: 55.00%

Epoch [9/10]
Train Loss: 1.1576 | Train Acc: 77.00%
Val Loss: 1.4604 | Val Acc: 57.00%

Epoch [10/10]
Train Loss: 1.0493 | Train Acc: 80.50%
Val Loss: 1.4223 | Val Acc: 57.00%

